In [1]:
# =======================
# GPU selection (no masking)
# =======================
import os
GPU_ID = 2  # <-- we want to run on cuda:2
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("CUDA_DEVICE_ORDER", "PCI_BUS_ID")

import sys, gc, random, warnings
warnings.filterwarnings("ignore")

import optuna
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

from gotabpfn import GraphFeatureOrdering, pidf_segpca, TabPFN25Head, TabPFN25Config
PIDFSegPCA = pidf_segpca

SEED = 42
DATA_FILE  = "SMK_CAN_187_combined_encoded.csv"
TARGET_COL = "Label"
N_TRIALS = 150

# ---- set runtime device to cuda:2 (do NOT mask others) ----
if torch.cuda.is_available():
    torch.cuda.set_device(GPU_ID)
    DEVICE_STR = f"cuda:{GPU_ID}"
else:
    DEVICE_STR = "cpu"

def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

def cleanup_cuda():
    gc.collect()
    if torch.cuda.is_available():
        try:
            torch.cuda.synchronize()
        except Exception:
            pass
        torch.cuda.empty_cache()

def ensure_multiclass_contiguous(y: np.ndarray):
    y = np.asarray(y).reshape(-1)
    uniq = np.unique(y)
    uniq_sorted = np.sort(uniq)
    class_map = {orig: i for i, orig in enumerate(uniq_sorted.tolist())}
    y_enc = np.vectorize(class_map.get, otypes=[np.int64])(y).astype(np.int64)
    return y_enc, class_map, int(len(class_map))

def compute_deltas_adjacent_corr_chunked(X_tr: np.ndarray, Pi_star: list[int], chunk: int = 2048, eps: float = 1e-12):
    X = torch.from_numpy(X_tr).float()  # CPU
    perm = torch.tensor(Pi_star, dtype=torch.long)

    n, d = X.shape
    deltas = torch.empty(d - 1, dtype=torch.float32)

    mu = X.mean(dim=0)
    Xc = X - mu
    std = Xc.std(dim=0, unbiased=False).clamp_min(eps)

    for a in range(0, d - 1, chunk):
        b = min(d - 1, a + chunk)
        i_idx = perm[a:b]
        j_idx = perm[a+1:b+1]
        Zi = (Xc[:, i_idx] / std[i_idx]).to(torch.float32)
        Zj = (Xc[:, j_idx] / std[j_idx]).to(torch.float32)
        corr = (Zi * Zj).mean(dim=0)
        deltas[a:b] = (1.0 - corr.abs()).cpu()

    return deltas

# -------- load --------
seed_everything(SEED)
df = pd.read_csv(DATA_FILE)
y_raw = df[TARGET_COL].to_numpy()
X_df  = df.drop(columns=[TARGET_COL])
y_all, class_map, NUM_CLASSES = ensure_multiclass_contiguous(y_raw)

scaler = StandardScaler()
X_all = scaler.fit_transform(X_df.values).astype(np.float32, copy=False)
X_all = np.asarray(X_all, dtype=np.float32, order="C")
y_all = np.asarray(y_all, dtype=np.int64)

print(f"[DATA] X={X_all.shape} | C={NUM_CLASSES} | map={class_map}")
print("[GPU ] cuda_available=", torch.cuda.is_available(),
      "| visible_gpus=", torch.cuda.device_count(),
      "| using_device=", DEVICE_STR)

rkf = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=SEED)

# Devices: explicitly cuda:2 (or cpu)
NSC_DEVICE = DEVICE_STR
TABPFN_DEVICE = DEVICE_STR

m_full = X_all.shape[1]
# ordering on a subset only (critical for SMK)
go_feat_grid = [g for g in [2000, 3000, 5000] if g <= m_full]
if len(go_feat_grid) == 0:
    go_feat_grid = [m_full]

def objective(trial: optuna.Trial):
    seed_everything(SEED)

    # -------- GO-LR once per trial, on subset --------
    go_metric = trial.suggest_categorical(
        "go_metric", ["correlation", "cosine", "manhattan", "euclidean"]  # drop KL for SMK huge-d
    )
    go_k = trial.suggest_int("go_num_clusters", 4, 12)
    go_refine_passes = trial.suggest_int("go_refine_passes", 1, 2)  # keep small
    go_direction = trial.suggest_categorical("go_direction_select", [True, False])
    go_feat_subsample = trial.suggest_categorical("go_feat_subsample", go_feat_grid)

    # deterministic feature subset for ordering
    rng = np.random.default_rng(SEED + 999)
    if go_feat_subsample < m_full:
        feat_idx = rng.choice(m_full, size=int(go_feat_subsample), replace=False)
        feat_idx.sort()
    else:
        feat_idx = np.arange(m_full)

    X_go = X_all[:, feat_idx]

    go = GraphFeatureOrdering(
        num_clusters=int(go_k),
        metric=go_metric,
        refine=True,
        direction_select=bool(go_direction),
        refine_passes=int(go_refine_passes),
    )

    # prefer GPU kmeans first, fallback CPU
    try:
        Pi_sub, _, _, _ = go.fit(X_go, seed=SEED, deterministic=True, use_cpu_kmeans=False)
    except Exception:
        cleanup_cuda()
        Pi_sub, _, _, _ = go.fit(X_go, seed=SEED, deterministic=True, use_cpu_kmeans=True)

    # map subset ordering back to full feature indices
    ordered_subset = feat_idx[np.array(Pi_sub, dtype=np.int64)].tolist()
    if go_feat_subsample < m_full:
        remaining = np.setdiff1d(np.arange(m_full), feat_idx, assume_unique=False)
        Pi_star = ordered_subset + remaining.tolist()
    else:
        Pi_star = ordered_subset

    # -------- NSC tune --------
    nsc_seg = trial.suggest_categorical("nsc_segmentation", ["uniform", "largest_jump"])  # keep simpler
    nsc_m_rule = trial.suggest_categorical("nsc_m_rule", ["default", "idf"])
    nsc_tau = trial.suggest_categorical("nsc_tau", [0.95, 0.99])
    nsc_gamma = trial.suggest_float("nsc_gamma", 1.0, 3.0)
    nsc_beta = trial.suggest_float("nsc_beta", 0.0, 0.9)
    nsc_Mmin = trial.suggest_categorical("nsc_Mmin", [16, 32, 48])
    nsc_Mmax = trial.suggest_categorical("nsc_Mmax", [128, 256, 384])  # reduced
    nsc_lmin = trial.suggest_categorical("nsc_lmin", [12, 16])
    assume_standardized = trial.suggest_categorical("assume_standardized", [True, False])

    tabpfn_seed = trial.suggest_categorical("tabpfn_seed", [0, 1, 2, 3, 4, 42])
    head_cfg = TabPFN25Config(
        task_type="binary",
        num_classes=int(NUM_CLASSES),
        device=TABPFN_DEVICE,   # "cuda:2" or "cpu"
        random_state=int(tabpfn_seed),
    )

    accs = []
    for fold_id, (tr_idx, va_idx) in enumerate(rkf.split(X_all, y_all), start=1):
        X_tr = X_all[tr_idx]; y_tr = y_all[tr_idx]
        X_va = X_all[va_idx]; y_va = y_all[va_idx]

        nsc = PIDFSegPCA(
            segmentation=nsc_seg,
            l_min=int(nsc_lmin),
            m_rule=nsc_m_rule,
            gamma=float(nsc_gamma),
            beta=float(nsc_beta),
            tau=float(nsc_tau),
            M_min=int(nsc_Mmin),
            M_max=int(nsc_Mmax),
            assume_standardized=bool(assume_standardized),
            device=NSC_DEVICE,   # "cuda:2" or "cpu"
        )

        deltas = None
        if nsc_seg != "uniform":
            deltas = compute_deltas_adjacent_corr_chunked(X_tr, Pi_star, chunk=2048)

        X_tr_t = torch.from_numpy(X_tr)
        nsc.configure(Pi_star=Pi_star, X_train=X_tr_t, tau=float(nsc_tau), deltas=deltas)

        Z_tr = nsc.compress(X_tr_t, mode="flatten").cpu().numpy()
        Z_va = nsc.compress(torch.from_numpy(X_va), mode="flatten").cpu().numpy()

        head = TabPFN25Head(head_cfg)
        head.fit(Z_tr, y_tr)

        P = head.predict_proba(Z_va)
        pred = np.argmax(P, axis=1)
        accs.append(float(accuracy_score(y_va, pred)))

        trial.report(float(np.mean(accs)), step=fold_id)
        if trial.should_prune():
            cleanup_cuda()
            raise optuna.TrialPruned()

        cleanup_cuda()

    mean_acc = float(np.mean(accs))
    std_acc = float(np.std(accs, ddof=1))
    trial.set_user_attr("mean_acc", mean_acc)
    trial.set_user_attr("std_acc", std_acc)
    return mean_acc

sampler = optuna.samplers.TPESampler(seed=SEED, multivariate=True, group=True)
pruner  = optuna.pruners.MedianPruner(n_warmup_steps=10)

study = optuna.create_study(direction="maximize", sampler=sampler, pruner=pruner)
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True, gc_after_trial=True, n_jobs=1)

best = study.best_trial
print("\n================ BEST TRIAL ================")
print(f"mean_acc ± std_acc = {best.user_attrs.get('mean_acc', best.value):.6f} ± {best.user_attrs.get('std_acc', float('nan')):.6f}")
print("\nBest hyperparameters:")
for k, v in best.params.items():
    print(f"  {k}: {v}")

[I 2026-01-23 19:31:34,627] A new study created in memory with name: no-name-8e7d03ef-77a2-4846-bd16-fae41177403f


[DATA] X=(187, 19993) | C=2 | map={0: 0, 1: 1}
[GPU ] cuda_available= True | visible_gpus= 8 | using_device= cuda:2


  0%|          | 0/150 [00:00<?, ?it/s]

[I 2026-01-23 19:54:42,413] Trial 0 finished with value: 0.7113798008534852 and parameters: {'go_metric': 'cosine', 'go_num_clusters': 5, 'go_refine_passes': 1, 'go_direction_select': False, 'go_feat_subsample': 3000, 'nsc_segmentation': 'uniform', 'nsc_m_rule': 'default', 'nsc_tau': 0.99, 'nsc_gamma': 2.049512863264476, 'nsc_beta': 0.3887505167779042, 'nsc_Mmin': 32, 'nsc_Mmax': 384, 'nsc_lmin': 12, 'assume_standardized': False, 'tabpfn_seed': 42}. Best is trial 0 with value: 0.7113798008534852.
[I 2026-01-23 20:14:59,587] Trial 1 finished with value: 0.6907254623044097 and parameters: {'go_metric': 'correlation', 'go_num_clusters': 7, 'go_refine_passes': 1, 'go_direction_select': True, 'go_feat_subsample': 2000, 'nsc_segmentation': 'largest_jump', 'nsc_m_rule': 'default', 'nsc_tau': 0.95, 'nsc_gamma': 2.8789978831283785, 'nsc_beta': 0.805344615384884, 'nsc_Mmin': 32, 'nsc_Mmax': 384, 'nsc_lmin': 12, 'assume_standardized': True, 'tabpfn_seed': 42}. Best is trial 0 with value: 0.711379